In [21]:
import os
import chromadb

from langchain_chroma import Chroma

from langchain_core.documents import Document

client = chromadb.Client()

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [22]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

In [23]:
from pathlib import Path
from dotenv import load_dotenv
import os
from groq import Groq

load_dotenv(dotenv_path=r"C:\Users\nakul\OneDrive\Desktop\training\.env")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
client = Groq(api_key=GROQ_API_KEY)

In [24]:
chromadb_client = chromadb.PersistentClient(path="./tesla_db")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [25]:
documents = [
    Document(
        id=1,
        page_content="We design, develop, manufacture, sell and lease high-performance fully electric vehicles and energy generation and storage systems, and offer services related to our products. We generally sell our products directly to customers, and continue to grow our customer-facing infrastructure through a global network of vehicle showrooms and service centers, Mobile Service, body shops, Supercharger stations and Destination Chargers to accelerate the widespread adoption of our products. We emphasize performance, attractive styling and the safety of our users and workforce in the design and manufacture of our products and are continuing to develop full self-driving technology for improved safety. We also strive to lower the cost of ownership for our customers through continuous efforts to reduce manufacturing costs and by offering financial and other services tailored to our products.",
        metadata={"year": 2023, "section": "business"}
    ),
    Document(
        id=2,
        page_content="We have previously experienced and may in the future experience launch and production ramp delays for new products and features. For example, we encountered unanticipated supplier issues that led to delays during the initial ramp of our first Model X and experienced challenges with a supplier and with ramping full automation for certain of our initial Model 3 manufacturing processes. In addition, we may introduce in the future new or unique manufacturing processes and design features for our products. As we expand our vehicle offerings and global footprint, there is no guarantee that we will be able to successfully and timely introduce and scale such processes or features.",
        metadata={"year": 2023, "section": "risk_factors"}
    ),
    Document(
        id=3,
        page_content="We recognize the importance of assessing, identifying, and managing material risks associated with cybersecurity threats, as such term is defined in Item 106(a) of Regulation S-K. These risks include, among other things: operational risks, intellectual property theft, fraud, extortion, harm to employees or customers and violation of data privacy or security laws. Identifying and assessing cybersecurity risk is integrated into our overall risk management systems and processes. Cybersecurity risks related to our business, technical operations, privacy and compliance issues are identified and addressed through a multi-faceted approach including third party assessments, internal IT Audit, IT security, governance, risk and compliance reviews. To defend, detect and respond to cybersecurity incidents, we, among other things: conduct proactive privacy and cybersecurity reviews of systems and applications, audit applicable data policies, perform penetration testing using external third-party tools and techniques to test security controls, operate a bug bounty program to encourage proactive vulnerability reporting, conduct employee training, monitor emerging laws and regulations related to data protection and information security (including our consumer products) and implement appropriate changes.",
        metadata={"year": 2023, "section": "cyber_security"}
    ),
    Document(
        id=4,
        page_content="The automotive segment includes the design, development, manufacturing, sales and leasing of high-performance fully electric vehicles as well as sales of automotive regulatory credits. Additionally, the automotive segment also includes services and other, which includes non-warranty after- sales vehicle services and parts, sales of used vehicles, retail merchandise, paid Supercharging and vehicle insurance revenue. The energy generation and storage segment includes the design, manufacture, installation, sales and leasing of solar energy generation and energy storage products and related services and sales of solar energy systems incentives.",
        metadata={"year": 2022, "section": "business"}
    ),
    Document(
        id=5,
        page_content="Since the first quarter of 2020, there has been a worldwide impact from the COVID-19 pandemic. Government regulations and shifting social behaviors have, at times, limited or closed non-essential transportation, government functions, business activities and person-to-person interactions. Global trade conditions and consumer trends that originated during the pandemic continue to persist and may also have long-lasting adverse impact on us and our industries independently of the progress of the pandemic.",
        metadata={"year": 2022, "section": "risk_factors"}
    ),
    Document(
        id=6,
        page_content="The German Umweltbundesamt issued our subsidiary in Germany a notice and fine in the amount of 12 million euro alleging its non-compliance under applicable laws relating to market participation notifications and take-back obligations with respect to end-of-life battery products required thereunder. In response to Tesla’s objection, the German Umweltbundesamt issued Tesla a revised fine notice dated April 29, 2021 in which it reduced the original fine amount to 1.45 million euro. This is primarily relating to administrative requirements, but Tesla has continued to take back battery packs, and filed a new objection in June 2021. A hearing took place on November 24, 2022, and the parties reached a settlement which resulted in a further reduction of the fine to 600,000 euro. Both parties have waived their right to appeal.",
        metadata={"year": 2022, "section": "legal_proceedings"}
    )
]

In [26]:
MODEL_NAME = "llama-3.3-70b-versatile"
TESLA_10K_COLLECTION = "tesla-10k-2019-to-2023"
HQ_COLLECTION = "hypothetical_questions"
CHROMA_DB_PATH = "./tesla_db"
RETRIEVER_K = 5

In [27]:
vectorstore_persisted.add_documents(documents=documents)

['1', '2', '3', '4', '5', '6']

In [28]:
from langchain_core.documents import Document

hypothetical_questions_system_message = """
Generate a list of exactly 3 hypothetical questions that the document presented in the input could be used to answer.
Generate only a list of questions, each question in a new line.
Do not number the questions or use bullet points.
Do not mention anything before or after the list.
"""

user_message_template = """
<Document>
{document}
</Document>
"""

hypothetical_questions = []

for document in documents:
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": hypothetical_questions_system_message},
                {"role": "user", "content": user_message_template.format(document=document.page_content)}
            ]
        )
        questions = response.choices[0].message.content.strip()
    except Exception as e:
        questions = ""

    for question in questions.split("\n"):
        hypothetical_questions.append(
            Document(
                page_content=question,
                metadata={
                    "parent_chunk_id": document.id,
                    "parent_collection": TESLA_10K_COLLECTION
                }
            )
        )

print(f"Generated {len(hypothetical_questions)} hypothetical questions")

Generated 6 hypothetical questions


In [34]:
vectorstore_persisted = Chroma(
    collection_name=TESLA_10K_COLLECTION,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding_model,
    client=chromadb_client,
    persist_directory=CHROMA_DB_PATH,
)

retriever = vectorstore_persisted.as_retriever(
    search_type="similarity",
    search_kwargs={"k": RETRIEVER_K},
)

hypothetical_questions_vectorstore = Chroma(
    collection_name=HQ_COLLECTION,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding_model,
    client=chromadb_client,
)

hq_retriever = hypothetical_questions_vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": RETRIEVER_K},
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [30]:
qna_system_message = """
You are an assistant to a financial services firm who answers user queries on annual reports.
User input will have the context required by you to answer user queries.
This context will be delimited by: <Context> and </Context>.
The context contains references to specific portions of a document relevant to the user query.

User queries will be delimited by: <Question> and </Question>.

Please answer user queries only using the context provided in the input.
Do not mention anything about the context in your final answer. Your response should only contain the answer to the question.

If the answer is not found in the context, respond "I don't know".
"""

qna_user_message_template = """
<Context>
Here are some documents that are relevant to the question mentioned below.
{context}
</Context>

<Question>
{question}
</Question>
"""

In [31]:
def retrieve_via_hypothetical_questions(user_query):
    hq_docs = hq_retriever.invoke(user_query)

    parent_ids = list(set([d.metadata["parent_chunk_id"] for d in hq_docs]))

    retrieved = vectorstore_persisted.get(ids=parent_ids)
    return retrieved["documents"]


def respond(user_query):
    context_list = retrieve_via_hypothetical_questions(user_query)
    context_for_query = "\n\n".join(context_list)

    prompt = [
        {"role": "system", "content": qna_system_message},
        {"role": "user", "content": qna_user_message_template.format(
            context=context_for_query,
            question=user_query,
        )},
    ]

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=prompt,
            temperature=0,
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Sorry, I encountered the following error:\n{e}"

In [33]:
hypothetical_questions_vectorstore.add_documents(documents=hypothetical_questions)
print(hypothetical_questions_vectorstore._collection.count())

6


In [35]:
user_query = "Any specific fines levied on the company in 2022?"
answer = respond(user_query)
print(f"Assistant: {answer}")

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


Assistant: Sorry, I encountered the following error:
Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'expired_api_key'}}
